# Humanitarian News Classification Notebook

This Jupyter Notebook demonstrates the process of classifying humanitarian news articles using machine learning models. It covers data preprocessing, model training, evaluation, and deployment steps.

## Key Sections
- **Data Exploration**: Loading and analyzing the dataset.
- **Preprocessing**: Cleaning and preparing text data for modeling.
- **Model Development**: Building and training classification models.
- **Evaluation**: Assessing model performance with metrics and visualizations.
- **Deployment**: Preparing the model for inference.

Repository: [JP-Thoma/Humanitarian-News-Classification](https://github.com/JP-Thoma/Humanitarian-News-Classification)  
Branch: main

*Note: Ensure all dependencies are installed before running the cells.*

In [1]:
# installing dependencies
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 35.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.0/625.0 kB 32.1 MB/s  0:00:00
  Attempting uninstall: idna0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/10 [joblib]
    Found existing installation: idna 3.11━━━━━━━━━━━━━━━━━━━━  2/10 [joblib]
    Uninstalling idna-3.11:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/10 [joblib]
      Successfully uninstalled idna-3.11━━━━━━━━━━━━━━━━━━━━━━  2/10 [joblib]
  Attempting uninstall: charset-normalizer━━━━━━━━━━━━━━━━━━━━━━━━  3/10 [idna]
    Found existing installation: charset-normalizer 3.4.5━━━━━  3/10 [idna]
    Uninstalling charset-normalizer-3.4.5:━━━━━━━━━━━━━━━━━━━━  3/10 [idna]
      Successfully uninstalled charset-normalizer-3.4.5━━━━━━━━━━━  5/10 [charset-normalizer]
  Attempting uninstall: certifi0m╺━━━━━━━━━━━━━━━━━━━  5/10 [charset-normalizer]
    Found existing installation: certifi 2026.2.25━━━━━━

In [2]:
# load api key and libraries
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("NEWSAPI_KEY")

if API_KEY is None:
    raise ValueError("API key not found. Please check your .env file.")

In [3]:
# intitialise NewsAPI Client

from newsapi import NewsApiClient

newsapi = NewsApiClient(api_key=API_KEY)

In [4]:
# specify keywords
QUERY = """
(
  outbreak OR epidemic OR pandemic OR cholera OR measles OR malaria OR dengue OR tuberculosis OR Ebola OR diphtheria OR malnutrition OR starvation OR famine OR "maternal mortality" OR trauma OR "mass casualty" OR "burn injuries" 
  OR "health system" OR "hospital overwhelmed"
)
AND
(
  war OR conflict OR violence OR bombardment OR airstrike OR siege OR displacement OR refugee OR IDP OR famine OR "food insecurity"
)
AND
(
  humanitarian OR NGO OR "aid workers" OR "medical charity"
)
"""

In [5]:
# specify country selection
MSF_COUNTRIES = [
    # Africa
    "Benin", "Burkina Faso", "Burundi", "Cameroon", "Central African Republic",
    "Chad", "Comoros", "Democratic Republic of Congo", "Eswatini", "Ethiopia",
    "Guinea", "Guinea-Bissau", "Ivory Coast", "Kenya", "Liberia",
    "Madagascar", "Malawi", "Mali", "Mauritania", "Mozambique",
    "Niger", "Nigeria", "Sierra Leone", "Somalia", "Somaliland",
    "South Africa", "South Sudan", "Sudan", "Tanzania", "Uganda", "Zimbabwe",

    # Middle East & North Africa
    "Egypt", "Iran", "Iraq", "Jordan", "Lebanon",
    "Libya", "Palestine", "Syria", "Yemen",

    # Asia & Pacific
    "Bangladesh", "Cambodia", "North Korea", "Hong Kong", "India",
    "Indonesia", "Kiribati", "Malaysia", "Myanmar", "Pakistan",
    "Papua New Guinea", "Philippines", "Thailand",

    # Europe & Central Asia
    "Afghanistan", "Armenia", "Azerbaijan", "Belarus", "Belgium",
    "Bulgaria", "France", "Greece", "Italy", "Kazakhstan",
    "Kyrgyzstan", "Latvia", "Lithuania", "Poland",
    "Russia", "Serbia", "Tajikistan", "Turkey",
    "Ukraine", "United Kingdom", "Uzbekistan",

    # Americas
    "Brazil", "Colombia", "El Salvador", "Guatemala",
    "Haiti", "Honduras", "Jamaica", "Mexico",
    "Nicaragua", "Panama", "United States", "Venezuela"
]

In [18]:
# fetch articles
articles_response = newsapi.get_everything(
    q=QUERY,
    from_param="2026-04-15",
    to="2026-04-25",
    language="en",
    sort_by="publishedAt",
    page_size=100
)

print(f"Total results: {articles_response['totalResults']}")
print(f"Articles retrieved: {len(articles_response['articles'])}")

articles_response['articles'][0]

Total results: 174
Articles retrieved: 91


{'source': {'id': None, 'name': 'Truthout'},
 'author': 'Nour Abo Aisha',
 'title': 'Lacking Proper Sanitation, Gaza’s Tent Camps Are Being Overrun by Rodents',
 'description': 'As conditions in Gaza’s tent camps worsen, rodents bring disease and destruction.',
 'url': 'https://truthout.org/articles/lacking-proper-sanitation-gazas-tent-camps-are-being-overrun-by-rodents/',
 'urlToImage': 'https://truthout.org/app/uploads/2026/04/GettyImages-2271926990-scaled.jpg',
 'publishedAt': '2026-04-24T17:04:09Z',
 'content': 'Part of the Series\r\nTruthout is a vital news source and a living history of political struggle. If you think our work is valuable, support us with a donation of any size.\r\nIn Gaza, displaced families… [+9863 chars]'}

In [19]:
# create dataframe
import pandas as pd

articles = articles_response['articles']

df = pd.json_normalize(articles)
df.head()

,author,title,description,url,urlToImage,publishedAt,content,source.id,source.name
0,Nour Abo Aisha,"Lacking Proper Sanitation, Gaza’s Tent Camps A...","As conditions in Gaza’s tent camps worsen, rod...",https://truthout.org/articles/lacking-proper-s...,https://truthout.org/app/uploads/2026/04/Getty...,2026-04-24T17:04:09Z,Part of the Series\r\nTruthout is a vital news...,NaN,Truthout
1,NaN,Trump’s War of Choice With Iran Threatens a Gl...,President Trump’s war of choice is generating ...,https://freerepublic.com/focus/f-news/4376287/...,NaN,2026-04-24T17:00:29Z,Skip to comments.\r\nTrumps War of Choice With...,NaN,Freerepublic.com
2,Muskan Singh,Quote of the Day by Michael Jackson: 'The most...,"Quote of the Day: Michael Jackson, the 'King o...",https://economictimes.indiatimes.com/news/inte...,"https://img.etimg.com/thumb/msid-130499018,wid...",2026-04-24T16:14:57Z,Quote of the Day: A powerful Quote of the Day ...,the-times-of-india,The Times of India
3,"Foreign, Commonwealth & Development Office",Conflict remains a leading cause of hunger: Mi...,Minister for Development Jenny Chapman gave a ...,https://www.gov.uk/government/speeches/conflic...,https://www.gov.uk/assets/frontend/govuk-openg...,2026-04-24T15:37:02Z,Thank you to everybody here for all that you d...,NaN,Www.gov.uk
4,Michael Arria,How the corporate media helped fuel Israel’s g...,Mondoweiss speaks with media critic Adam Johns...,https://mondoweiss.net/2026/04/how-the-corpora...,https://mondoweiss.net/wp-content/uploads/2023...,2026-04-24T15:15:00Z,"In his new book, How to Sell a Genocide: The M...",NaN,Mondoweiss


In [20]:
# select relevant columns
df = df[[
    "source.name",
    "author",
    "title",
    "description",
    "url",
    "publishedAt",
    "content"
]]

df.rename(columns={
    "source.name": "source"
}, inplace=True)

print(f"Final dataset size: {len(df)}")
df.head()

Final dataset size: 91


,source,author,title,description,url,publishedAt,content
0,Truthout,Nour Abo Aisha,"Lacking Proper Sanitation, Gaza’s Tent Camps A...","As conditions in Gaza’s tent camps worsen, rod...",https://truthout.org/articles/lacking-proper-s...,2026-04-24T17:04:09Z,Part of the Series\r\nTruthout is a vital news...
1,Freerepublic.com,NaN,Trump’s War of Choice With Iran Threatens a Gl...,President Trump’s war of choice is generating ...,https://freerepublic.com/focus/f-news/4376287/...,2026-04-24T17:00:29Z,Skip to comments.\r\nTrumps War of Choice With...
2,The Times of India,Muskan Singh,Quote of the Day by Michael Jackson: 'The most...,"Quote of the Day: Michael Jackson, the 'King o...",https://economictimes.indiatimes.com/news/inte...,2026-04-24T16:14:57Z,Quote of the Day: A powerful Quote of the Day ...
3,Www.gov.uk,"Foreign, Commonwealth & Development Office",Conflict remains a leading cause of hunger: Mi...,Minister for Development Jenny Chapman gave a ...,https://www.gov.uk/government/speeches/conflic...,2026-04-24T15:37:02Z,Thank you to everybody here for all that you d...
4,Mondoweiss,Michael Arria,How the corporate media helped fuel Israel’s g...,Mondoweiss speaks with media critic Adam Johns...,https://mondoweiss.net/2026/04/how-the-corpora...,2026-04-24T15:15:00Z,"In his new book, How to Sell a Genocide: The M..."


In [21]:
# light cleaning
# Drop duplicates based on title
df.drop_duplicates(subset="title", inplace=True)

# Drop rows with no description (important for later classification)
df = df[df["description"].notna()]

df.reset_index(drop=True, inplace=True)

print(f"Final dataset size: {len(df)}")

Final dataset size: 90
